# report07 — SBR — 표적을 조준해 밝기를 계산하다

> ### ❓ 이 리포트가 답하는 질문
> **Sionna 가 못 내는 RCS 를 어떻게 계산했나?**

### ⚡ 결론부터 (TL;DR)

1. **레이더가 표적을 잡으려면 표적이 '밝아야' 한다.** 그 밝기가 RCS(레이더 되비침 밝기, σ) 다. 그런데 전파 시뮬레이터 Sionna 의 기본 광선엔진에는 밝기를 만드는 **적분 단계가 없어** σ 를 못 준다(왜 그런지는 앞 리포트 06). 그래서 **그 광선엔진 위에 빠진 적분을 얹었다** — 이게 **SBR**(광선을 쏘고 튕겨 밝기를 재는 방법)이고, FEKO·CST 같은 상용 전자기 솔버가 큰 표적에 쓰는 표준이다.
2. **이게 이 프로젝트의 핵심 기여다.** Sionna 는 **통신(Communication)** 시뮬레이터라 표적을 되쏘는 밝기(**센싱, Sensing** 의 핵심)가 없다(리포트 06). SBR 로 그 밝기를 얹으면 통신 도구가 **센싱까지 하는 ISAC(통신·센싱 통합) 시뮬레이터**가 된다 — **ISAC = C(Sionna) + S(우리 SBR).** 다만 정직히: SBR 도 Sionna 의 전파 솔버가 아니라 그 밑엔진 **Mitsuba**(광선추적 엔진 — 3D 장면에서 광선이 어느 삼각형에 부딪히나 계산하는 그래픽스 라이브러리; 영화 CGI·블렌더와 같은 부류이고 Sionna 는 이 위에 지어졌다) + **우리 물리(PO)** 로 계산한다.
3. **SBR 은 두 가지를 바꾼다.** ① 광선을 사방에 흩뿌리지 않고 **표적을 정면 조준**해 촘촘한 평행 격자로 쏜다 → 거의 다 맞는다. ② 맞은 점마다 **레이더로 되돌아올 양을 PO(물리광학)로, 위상까지 맞춰 계산**해 전부 더한다 → 광선이 물리적으로 튕겨 수신기에 닿기를 기다리지 않고 그 자리에서 셈한다.
4. **답을 아는 물건으로 검증했다.** @3.5 GHz(파장 λ = 8.57 cm)에서 금속 평판(교과서 값 σ = 4πA²/λ²)은 오차 **-0.01 dB**, 금속구(σ = πr²)는 **+0.39 dB**. 완전히 따로 돌린 report6 의 같은 커널도 각각 **-0.01 / +0.39 dB** 로 같은 값을 낸다.
5. **옛 방식(가림 없는 순수 PO)보다 밝기가 낮게 나온다 — 그게 맞다.** DJI Mavic 4 Pro 한 방위에서, 순수 PO 가 '빛나는 면' 이라 센 18,409 조각 중 **32% 가 앞 부품에 가려져** 실제로는 안 보였다. 가려진 면을 빼자 방위평균 밝기가 **-16.46 → -19.66 dBsm** 으로 **3.20 dB** 내려간다.

### 🗺️ 어디부터 읽나

| 절 | 무엇을 |  |
|---|---|---|
| §1 | RCS 가 뭐고 왜 디텍션에 필수인가 — 밝아야 잡힌다 | 한 문단, 비유 |
| §2 | **SBR 이 무엇인가** — 조준해서 쏘고, 되쏘는 양을 더한다 | 이 리포트의 핵심 |
| §3 | **검증** — 평판·구의 교과서 값과 대조 | 정말 맞나? |
| §4 | **가림** — 뒤에 숨은 면을 빼면 밝기가 내려간다 | SBR 이 실제로 고치는 것 |
| 바쁘면 | §4 그림 하나 (report2_occlusion.png) | 한 장으로 요점 |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 검증 기준 (정답) | 교과서 폐형식 — 금속구 σ = πr² (광학영역) · 금속평판 σ = 4πA²/λ² (정면 조사) | 📐 해석해 |
| SBR 검증·격자수렴·가림 측정값 | **`outputs/report2_waveform_rcs.json`** 의 `sbr_validation` / `occlusion` (그림 `report2_sbr_validate.png` · `report2_occlusion.png` 과 같은 소스) | 🟡 측정 (SBR = 우리 구현, Mitsuba 광선) |
| 같은 커널의 독립 재현 | **`outputs/report6_sbr.json`** 의 `kernel` — 동일 해석해 검증을 따로 돌린 값 | 🟡 교차확인 |
| SBR 구현 | **`src/rcs_sbr.py`** — Mitsuba 3 / OptiX 광선 + PO 표면적분. **Sionna 가 쓰는 그 광선엔진 그대로** | 🟡 우리 구현 (Sionna 에 RCS 솔버가 없음) |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `po` | 순수 물리광학 (`src/rcs_po.py`) — 점구름 PO. **가림 없음** | 🔴 **별도** (numpy, CPU). **비교·검증용으로만** 남겨둠 — 기본 엔진은 SBR |
| `sionna-render` | Sionna RT `Scene.render_to_file()` — 씬·**추적된 광선**·라디오맵을 사진처럼 렌더 | 🟢 **Sionna 내부** (Mitsuba 3 경로추적 렌더러, GPU) |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `mitsuba` | 3.8.0 | Sionna RT 의 렌더러·광선추적 백엔드 (OptiX, GPU). SBR 도 이걸 쓴다 |
| `drjit` | 1.3.1 | Mitsuba 의 JIT 컴파일러 — GPU 커널 생성 |
| `trimesh` | 4.12.2 | 메쉬 CAD·**검증** — 로프트/스윕/불리언 + watertight·법선·퇴화면 검사 |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: SBR 커널 검증은 GPU 한 장에서 수십 초. 광선격자 λ/16, Mitsuba 3 / OptiX.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2

# 커널 해석해 검증(구 πr² / 평판 4πA²/λ²) + 격자 수렴
~/.venvs/py312/bin/python src/rcs_sbr.py

# 측정 + 그림 + JSON (sbr_validation / occlusion 을 남긴다)
~/.venvs/py312/bin/python src/viz_report2.py

# JSON -> report07.ipynb (이 파일)
~/.venvs/py312/bin/python src/make_notebook07.py
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report2_waveform_rcs.json` | **이 노트북의 SBR 숫자.** sbr_validation / occlusion 블록 |
| `outputs/report6_sbr.json` | 같은 커널의 독립 재현 (kernel 블록) |
| `outputs/figures/report2_sbr_validate.png` | §3 SBR 검증 · 격자 수렴 |
| `outputs/figures/report2_occlusion.png` | §4 가림 — 순수 PO vs SBR |
| `outputs/figures/report2_po_vs_sbr.png` | §4 방위 패턴 비교 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **절대 RCS 값을 보장하지 않는다.** SBR 은 해석해(구·평판)로만 검증됐고 드론 실측 앵커링이 없다. 이 리포트가 지지하는 것은 **방법의 정합성**과 **가림의 방향·규모**이지, 특정 드론의 절대 dBsm 이 아니다.
- **매끈한 구의 단일 격자 값은 수렴한 숫자가 아니다.** 구는 되쏘는 지점이 딱 하나뿐이라 격자를 어디에 맞추느냐에 민감하다(λ/8 산포 5.3 dB). 드론처럼 되쏘는 점이 많은 표적은 방위평균이 저절로 안정된다 — §3 에서 짚는다.
- **오목한 곳의 다중반사(2·3차) 값은 규모(작다)만 신뢰한다.** SBR 이 실제로 고치는 것은 가림이고, 다중반사는 부차적이고 작은 항이다. 그 정확한 dB 값은 이 리포트의 주장이 아니다.
- **SBR 은 유전체 투과를 모른다** — 광선을 첫 충돌에서 끝내므로 준투명 플라스틱 셸을 지나는 전파는 못 본다. 이 성질이 재질 실험에 주는 영향은 report08 에서 다룬다.
- **표적이 밝아야 잡힌다 — RCS 는 디텍션에 필수다.** 다만 그 밝기를 Sionna 기본 광선으론 낼 수 없어(산란적분 부재) SBR 로 따로 계산할 뿐이다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| **앞** — [report06](report06.ipynb) | Sionna 의 한계 — 왜 기본 광선엔진이 표적 밝기(σ)를 못 내나 |
| **다음** — [report08](report08.ipynb) | 이 SBR 로 잰 **실제 드론들의 밝기(RCS) 결과** |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **RCS (σ)** | 레이더 되비침 밝기 [m²]. 표적이 레이더 쪽으로 얼마나 세게 되쏘는가. 밝을수록 잡기 쉽다. dBsm = 10·log₁₀(σ / 1 m²) |
| **σ 를 넓이로** | 밝기는 결국 **되쏘는 표면의 넓이**에서 나온다 — 넓은 판이 작은 못보다 밝게 반짝인다 |
| **dBsm** | 1 m² 를 기준으로 한 데시벨 단위. −20 dBsm ≈ 0.01 m² |
| **PO (물리광학)** | 표적 표면의 작은 조각들이 되쏘는 양을 **위상까지 맞춰 다 더하는 계산**. 밝기(σ)는 이 덧셈에서 나온다 |
| **SBR** | Shooting-and-Bouncing Rays. 표적을 **조준해 광선을 쏘고**, 맞은 면이 레이더로 **되쏘는 양을 PO 로 계산·합산**한다. 상용 전자기 솔버의 표준 |
| **GO (기하광학)** | 표면을 '점 거울' 로만 본다. 벽·바닥에서 어디로 튕기는지는 정확하지만 **넓이 항이 없어 밝기를 못 준다** — 전파 광선추적이 이것 |
| **가림(occlusion)** | 앞의 면에 막혀 실제로는 레이더에 안 보이는 면. 이걸 안 빼면 밝기를 부풀린다 |
| **정반사점** | 광선이 거울처럼 **똑바로 되돌아오는 딱 한 지점**. 매끈한 곡면은 이게 하나뿐 |
| **격자밀도 λ/d** | 쏘는 평행 광선을 파장(λ)당 몇 발로 촘촘히 하느냐. λ/16 = 파장 한 칸에 16 발 |
| **위상** | 파동이 되쏠 때의 타이밍. 같은 타이밍으로 겹치면 밝아지고 엇갈리면 상쇄된다 |

</details>

---


## 🔰 5분이면 이해하는 이 리포트

**한 줄:** 레이더는 표적이 **밝게 되비출수록** 잘 잡습니다. 그 밝기가 RCS 입니다. 그런데 전파 시뮬레이터(Sionna)의 광선은 이 밝기를 혼자서는 못 잽니다. 그래서 그 광선엔진 **위에** 밝기를 만드는 계산 하나를 얹었습니다.

**비유로.** 어두운 방에서 손전등으로 물건을 비춰, 그 물건이 내 쪽으로 얼마나 되쏘는지 알고 싶다고 합시다.

- 옛날 방식은 광선을 **사방에 마구 흩뿌려** 놓고 그중 몇 개가 우연히 물건에 맞아 되튕겨 나에게 돌아오길 기다립니다. 표적이 작으면 거의 다 빗나가서 **복권 당첨을 기다리는 꼴**입니다.
- SBR 은 대신 **손전등 빔으로 물건을 통째로 덮습니다.** 물건을 정면 조준해 촘촘한 평행 광선 다발을 쏘니 거의 다 맞습니다 — 복권이 아닙니다.
- 그리고 물리적으로 되튕겨 돌아오길 기다리는 대신, **맞은 밝은 점 하나하나가 내 쪽으로 얼마나 되쏘는지를 그 자리에서 계산해 전부 더합니다.** 이 합이 곧 물건의 밝기입니다.
- 덤: 광선은 **맨 앞에 처음 부딪힌 곳**에서 멈춥니다. 그러니 뒤에 가려진 부분은 저절로 빠집니다. 이 공짜 '가림' 이 이 방법의 진짜 이득입니다.

**확인한 것 두 가지.**
1. **맞는가?** — 밝기를 이미 아는 물건(금속 평판·금속구)에 대고 재보니 교과서 공식과 **0.01~0.4 dB** 안에서 맞았습니다.
2. **뭘 고치나?** — 가림을 안 하던 옛 방식은 **뒤에 숨어 안 보이는 면까지 세어** 밝기를 부풀리고 있었습니다. 숨은 면을 빼자 밝기가 약 3 dB 내려갔습니다.

> 한 마디로: **Sionna 를 버리는 게 아니라, 그 광선엔진 위에 빠진 밝기 계산을 얹은 것**입니다.

> 📎 **앞 리포트([report06](report06.ipynb))** 는 Sionna 의 기본 광선이 왜 밝기(σ)를 못 내는지를 보였습니다. 이 리포트는 그 **빈자리를 메우는 방법**입니다.

## 1. 왜 밝기(RCS)가 필요한가 — 밝아야 잡힌다

레이더는 결국 **되돌아온 메아리의 세기**로 표적을 알아챕니다. 표적이 어두우면(RCS 가 작으면) 메아리가 주변 잡음에 묻혀 검출이 실패합니다. 그러니 **표적이 얼마나 밝게 되비추는가**(RCS, σ)를 아는 것은 디텍션의 출발점입니다.

비유하면, 밤길에서 반사조끼를 입은 사람은 잘 보이고 검은 옷을 입은 사람은 안 보입니다. RCS 는 표적이 얼마나 '반사조끼' 인가를 넓이 단위(m²)로 매긴 값입니다.

문제는 이 밝기를 어떻게 계산하느냐입니다. 전파 시뮬레이터의 기본 광선엔진은 광선이 벽·바닥에서 **어디로 튕기는지**는 정확히 알지만, 표면이 **얼마나 넓은지**를 셈에 넣지 못합니다. 되비침의 밝기는 넓이에서 나오는데(넓은 판이 작은 못보다 밝게 반짝이죠) 그 넓이 항이 식에 없는 것입니다.

> → 왜 기본 광선엔진에 넓이 항이 없는지, 그 증거는 **앞 리포트 [report06](report06.ipynb)** 이 다룹니다. 여기서는 "그래서 무엇을 얹어 밝기를 냈나" 로 바로 갑니다.

## 2. SBR — 조준해서 쏘고, 되쏘는 양을 더한다

SBR(Shooting-and-Bouncing Rays)은 밝기를 내기 위해 **두 가지**를 바꿉니다.

**① 사방에 뿌리지 않고 표적을 정면 조준한다.** 광선을 아무 방향으로나 흩뿌리면 작은 표적은 대부분 빗나갑니다 — 되돌아오는 몇 발을 기다리는 건 복권과 같습니다. 대신 시선 방향 $\hat u$ 에서 **간격이 촘촘한 평행 광선 격자**를 표적에 곧장 쏩니다. 그러면 거의 모든 광선이 표적의 어딘가에 맞습니다.

**② 물리적으로 되튕겨 오길 기다리지 않고, 되쏘는 양을 그 자리에서 계산한다.** 광선이 맞은 점 $\vec p_i$ 마다, 그 면이 레이더 쪽으로 되쏘는 양을 PO(물리광학)로 — **위상까지 맞춰** — 셈해 전부 더합니다:

$$E(\hat u)=\sum_{\text{맞은 점}} |\Gamma_i|\; e^{\,j\,2k\,\vec p_i\cdot\hat u}\; d^2,\qquad \sigma=\frac{4\pi}{\lambda^2}\,|E|^2$$

- $|\Gamma_i|$ = 그 점 재질이 되쏘는 세기(반사계수), $d$ = 광선 격자 간격($d^2$ = 광선 한 발이 대표하는 넓이), $k=2\pi/\lambda$ = 파수.
- $e^{\,j2k\,\vec p_i\cdot\hat u}$ 는 **위상**입니다 — 각 점의 되쏨이 같은 타이밍으로 겹치면 밝아지고 엇갈리면 상쇄됩니다. 이 위상 덧셈이 곧 표면 넓이 위의 적분과 같아집니다.

> **쉽게 말하면 —** 손전등 빔으로 표적을 통째로 덮고(①), 각 밝은 점이 내 쪽으로 되쏘는 양을 계산해 합산합니다(②). 그 합이 표적의 밝기입니다.

그리고 **가림이 공짜로 따라옵니다.** 광선은 맨 앞에 처음 부딪힌 곳에서 멈추므로, 뒤에 가려진 면은 애초에 합에 끼지 못합니다(§4 가 이 이득을 잽니다).

**이것이 SBR 이고**, FEKO·CST 같은 상용 전자기(EM) 솔버가 큰 표적에 쓰는 **표준 방법**입니다. 우리 구현은 `src/rcs_sbr.py` 이며, 광선추적은 **Sionna 가 쓰는 Mitsuba 3 / OptiX 엔진 그대로** GPU 에서 돕니다. 새 시뮬레이터를 만든 게 아니라 **광선엔진 위에 빠진 밝기 계산을 얹은 것**입니다.

![.](outputs/renders/anim/paths_build.gif)

<sub>Sionna(Mitsuba) 광선이 반사를 거듭하며 늘어나는 모습 — SBR 은 이 광선으로 표적을 조준한다.</sub>

## 3. 검증 — 답을 아는 물건에 대고 재보기

새 계산이 옳은지 확인하는 가장 정직한 방법은 **답이 이미 알려진 물건**에 대고 재보는 것입니다. 레이더 교과서는 두 물건의 밝기를 폐형식(닫힌 공식)으로 줍니다: 정면으로 조사한 **금속 평판**은 σ = 4πA²/λ², **금속구**는 σ = πr².

![sbr validate](outputs/figures/report2_sbr_validate.png)

**@ 3.5 GHz (λ = 8.57 cm) 대조:**

| 표적 | 교과서 값 | SBR 오차 | report6 독립재현 |
|---|---|---|---|
| 금속 평판 0.4 × 0.4 m (정면) | +16.42 dBsm | **-0.01 dB** (λ/6) | -0.01 dB |
| 금속구 r = 0.5 m | -1.05 dBsm | **+0.39 dB** (λ/10) | +0.39 dB |

→ 두 물건 모두 교과서 값과 **1 dB 이내**로 맞고, 완전히 따로 돌린 report6 커널도 같은 값을 냅니다. **방법 자체는 옳습니다.**

**단, 매끈한 구는 원래 까다로운 표적입니다 — 정직하게 짚습니다.** 구는 레이더 쪽으로 **똑바로 되쏘는 지점(정반사점)이 딱 하나뿐**이라, 촘촘한 광선 격자 중 **한 발이 그 점 위에 떨어지느냐**에 따라 값이 흔들립니다. 격자 밀도는 그대로 두고 격자를 살짝 옆으로 옮겨가며 재보면(그림 b):

| 격자 | 정렬만 흔들었을 때 산포 |
|---|---|
| λ/8 | **5.28 dB** |
| λ/12 | **1.37 dB** |
| λ/16 | **1.78 dB** |
| λ/24 | **0.34 dB** |

→ 거친 λ/8 에서는 격자 위치만으로 **5.3 dB** 가 움직이지만, 촘촘한 λ/24 에서는 0.34 dB 로 닫힙니다. 즉 매끈한 곡면은 촘촘한 격자에서만 안정됩니다.

**하지만 드론은 성질이 다릅니다** — 되쏘는 점이 몸통·팔·모터·프로펠러에 잔뜩 흩어져 있어, 방위(각도)를 돌려가며 평균을 내면 **저절로 안정**됩니다(그림 c):

| 격자 | λ/6 | λ/8 | λ/12 | λ/16 | λ/24 |
|---|---|---|---|---|---|
| LTE 1.8 GHz 방위평균 [dBsm] | -18.67 | -19.81 | -19.81 | -19.68 | -19.70 |
| 5G NR 3.5 GHz 방위평균 [dBsm] | -18.59 | -19.64 | -19.66 | -19.75 | -19.80 |

→ λ/8 부터는 촘촘한 λ/24 값과 **0.16 dB 이내**로 붙습니다(가장 거친 λ/6 만 1.20 dB 벗어납니다). **드론 측정은 λ/16 로 돌리므로 수렴 구간 안**입니다.

> 정리하면, 정반사점이 하나뿐인 매끈한 구가 이 방법에게 가장 까다로운 표적이고, 되쏘는 점이 많은 드론은 방위평균이 스스로 값을 안정시켜 줍니다 — 표적의 성질 차이입니다.

## 4. 가림 — 뒤에 숨은 면을 빼면 밝기가 내려간다

가림을 처리하지 않는 옛 방식(순수 PO)은 표면 조각이 시선 쪽을 향하기만 하면($\hat n\cdot\hat u > 0$) '빛나는 면' 으로 셉니다 — **앞의 부품에 완전히 가려져 실제로는 레이더에 안 보여도** 그렇습니다. SBR 은 광선을 실제로 쏘므로, 뒤에 숨은 면은 광선이 앞면에서 멈춰 애초에 닿지 못합니다.

![occlusion](outputs/figures/report2_occlusion.png)

DJI Mavic 4 Pro 를 한 방위(방위각 30°, 올려본각 15°)에서 들여다보면:

| | 조각 수 | 투영 넓이 |
|---|---|---|
| 순수 PO 가 '빛난다' 고 센 면 | **18,409** | 880 cm² |
| └ 그중 **실제로는 뒤에 가려진** 면 | **5,820 (32%)** | — |
| 광선이 **실제로 맞은** 면 (SBR) | 12,589 | **490 cm²** |

→ 순수 PO 는 실제로 보이는 것보다 **약 1.8 배 넓은 면적**을 밝기에 넣고 있었습니다. 가려진 면을 빼면 밝기가 그만큼 내려갑니다:

**DJI Mavic 4 Pro, 72 방위 평균, 올려본각 15°, 3.5 GHz:**

| 방식 | 방위평균 밝기 | 차이 |
|---|---|---|
| 순수 PO (가림 없음) | -16.46 dBsm | — |
| **SBR (가림 처리)** | **-19.66 dBsm** | **-3.20 dB** ← 가림 |
| SBR + 오목부 다중반사 | -19.56 dBsm | +0.10 dB ← 다중반사 |

![po vs sbr](outputs/figures/report2_po_vs_sbr.png)

흥미롭게도, PO 계열에 흔히 지적되는 '오목한 곳의 다중반사를 놓친다' 는 약점은 여기서 **+0.10 dB** 짜리 사소한 항이었습니다. 실제로 밝기를 바꾼 것은 **가림(3.20 dB)** 이었습니다 — 한 자릿수 차이입니다.

> 이것이 SBR 이 순수 PO 대비 실제로 고치는 지점입니다. 광선을 쏘아 **레이더에 실제로 보이는 면만** 밝기에 넣으니, 뒤에 숨은 면을 세던 옛 밝기보다 값이 낮아집니다.

In [ ]:
# §3·§4 재현 — SBR 커널 검증(교과서 값) + 가림 대조
import rcs_sbr
rcs_sbr.validate(3.5e9)        # 금속구 πr² / 평판 4πA²/λ² 대조 + 격자 수렴
rcs_sbr.compare_with_po()      # 순수 PO vs SBR (가림의 대가)

In [ ]:
# SBR 커널을 한 방향에서 직접 호출 — 광선 격자를 쏘고(가림 처리), 위상 맞춰 더해 밝기를 낸다.
import numpy as np
from rcs_po import drone_rcs_pattern_bw, dbsm    # 기본 엔진이 'sbr' 이다

az = np.arange(0, 361, 2.0)
sig, n_rays = drone_rcs_pattern_bw('mavic4pro', 3.5e9, 100e6, az, el_deg=15.0, n_f=5)
print(f'방위당 광선 {n_rays:,}발  (격자 λ/16)')
print(f'방위평균 밝기 {dbsm(np.mean(sig)):+.2f} dBsm')
# ↑ 드론별 밝기 '결과'의 해석은 다음 리포트(report08) 소관이다.

---
## 📌 정리

### ❓ "Sionna 가 못 내는 RCS 를 어떻게 계산했나?"

**표적을 정면 조준해 촘촘한 평행 광선으로 덮고, 맞은 점마다 레이더로 되쏘는 양을 위상까지 맞춰 더했습니다.** 그 합이 곧 밝기(RCS)입니다. 이게 SBR 이고, 상용 전자기 솔버가 쓰는 표준입니다. Sionna 를 버린 게 아니라 **그 광선엔진 위에 빠진 밝기 계산을 얹은 것**입니다.

**옳은가?** 금속 평판(-0.01 dB)·금속구(+0.39 dB) 교과서 값과 **1 dB 이내**로 맞았고, 따로 돌린 커널도 같은 값을 냈습니다. 매끈한 구가 격자 정렬에 민감하다는 성질까지 짚었으며, 드론은 방위평균이 저절로 안정돼 **λ/16 수렴 구간**에 있습니다.

**무엇을 고치나?** 순수 PO 는 뒤에 숨어 안 보이는 면까지 세어 밝기를 부풀렸습니다. 가려진 면(이 방위에서 32%)을 빼자 밝기가 **-16.46 → -19.66 dBsm** 으로 내려갔습니다. 다중반사가 아니라 **가림이 진짜 이득**이었습니다.

### ⚠️ 이 리포트가 보장하지 않는 것
- **절대 RCS 값** — 해석해로만 검증했고 드론 실측 앵커가 없습니다. 방법의 정합성과 가림의 방향·규모만 주장합니다.
- **매끈한 구의 단일 격자 값**, **다중반사의 정확한 dB** — 인용하지 마십시오(§3·§4).
- **유전체 투과** — SBR 은 첫 충돌에서 멈춰 준투명 셸을 지나는 전파를 못 봅니다(→ report08).

> 표적이 밝아야 잡힙니다 — RCS 는 디텍션에 필수입니다. 다만 그 밝기를 Sionna 기본 광선으론 낼 수 없어(산란적분 부재) SBR 로 따로 계산할 뿐입니다.

---
**다음 리포트**: [report08](report08.ipynb) — 이 SBR 로 잰 **실제 드론 5종의 밝기(RCS) 결과**.